In [1]:
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import scipy.io as sio
import os
import liana as li
from liana.method import cellphonedb, geometric_mean

# seed for reproducibility
RNG_SEED = 1337
np.random.seed(RNG_SEED) 

export_dir = "output/03_RCTD_SpotDeconvolution/ref_export"   # written by 03a_RCTD_spotdeconvolution.qmd
out_dir    = "output/03b_runLiana_scRNAseq"
os.makedirs(out_dir, exist_ok=True)

counts = sio.mmread(f"{export_dir}/lognormalized_matrix.mtx").tocsr().T
genes = pd.read_csv(f"{export_dir}/features.tsv", header=None)[0].values
barcodes = pd.read_csv(f"{export_dir}/barcodes.tsv", header=None)[0].values
meta = pd.read_csv(f"{export_dir}/metadata.csv").set_index("barcode").loc[barcodes]

adata = ad.AnnData(X=counts, obs=meta, var=pd.DataFrame(index=genes))
adata.var_names_make_unique()
adata.raw = adata   # already log-normalized from R

# candidate_resource.csv now pools the significant candidate LR pairs from all 4
# spatial pipelines (Astrocyte-Vascular, Vascular-Microglia, OPC-Cajal-Retzius,
# Neurogenic_lineage-GC) — see the R "Save data for LIANA+ CCC" section in 03a. Running
# the permutation test once on the pooled resource, then filtering per cell-type
# pair below, avoids paying the (expensive) permutation cost 4 times.
candidate_resource = pd.read_csv(f"{export_dir}/candidate_resource.csv")

# mirror ref_seurat$Celltype_updated from 03a_RCTD_spotdeconvolution.qmd exactly
# (build-full-reference chunk), so LIANA's "Celltype" groups match the RCTD
# cell types used throughout the spatial pipeline:
#   ref_seurat$Celltype_updated <- ref_seurat$Celltype %>%
#     gsub("qNSC|nIPC|Neuroblast","Neurogenic_lineage", .) %>%
#     gsub("SMC|Pericyte|Endothelial","Vascular", .)
celltype_merge_map = {
    "qNSC": "Neurogenic_lineage",
    "nIPC": "Neurogenic_lineage",
    "Neuroblast": "Neurogenic_lineage",
    "SMC": "Vascular",
    "Pericyte": "Vascular",
    "Endothelial": "Vascular",
}
adata.obs["Celltype"] = adata.obs["Celltype"].replace(celltype_merge_map)

adata.obs["Celltype"].value_counts()  


methods = [cellphonedb, geometric_mean]
custom_aggregate = li.mt.AggregateClass(li.mt.aggregate_meta, methods=methods)

results = []
for age_label in ["Young", "Middle", "Old"]:
    sub = adata[adata.obs["Age"] == age_label].copy()

    custom_aggregate(sub,
                      groupby="Celltype",
                      resource=candidate_resource,
                      expr_prop=0.1,
                      n_perms=1000,
                      seed=RNG_SEED,
                      verbose=True,
                      key_added="cpdb_gmean_agg")

    res_age = sub.uns["cpdb_gmean_agg"].copy()
    res_age["Age"] = age_label
    results.append(res_age)

res_by_age = pd.concat(results, ignore_index=True)


/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: 

Using provided `resource`.


11004 features of mat are empty, they will be removed.
Converting `Celltype` to categorical!
/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:307: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:167: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.01 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 5382 samples and 464 features
Running CellPhoneDB


100%|█████████████████████████████████████| 1000/1000 [00:09<00:00, 103.18it/s]


...

9461 features of mat are empty, they will be removed.
Converting `Celltype` to categorical!
/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:307: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:167: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.00 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 20259 samples and 469 features
Running CellPhoneDB


100%|██████████████████████████████████████| 1000/1000 [00:20<00:00, 49.26it/s]


Running Geometric Mean


100%|██████████████████████████████████████| 1000/1000 [00:20<00:00, 48.09it/s]


In [14]:
# res_by_age already has every cell-type pair present in the data (groupby="Celltype"
# runs CCC across all pairs at once) — extract_pair() just filters the one shared
# result table per pair of interest, both interaction directions included.

def extract_pair(res_df, ct1, ct2):
    return res_df[
        ((res_df["source"] == ct1) & (res_df["target"] == ct2)) |
        ((res_df["source"] == ct2) & (res_df["target"] == ct1))
    ].copy()

def finalize_and_save(res_pair, filename):
    res_pair = res_pair.rename(columns={"Age": "age"})
    res_pair["age"] = res_pair["age"].str.lower()
    res_pair.to_csv(os.path.join(out_dir, filename), index=False)
    return res_pair

res_astro_vasc_by_age    = finalize_and_save(extract_pair(res_by_age, "Astrocyte", "Vascular"),
                                              "liana_astro_vasc_by_age.csv")
res_vasc_microglia_by_age = finalize_and_save(extract_pair(res_by_age, "Vascular", "Microglia"),
                                               "liana_vasc_microglia_by_age.csv")
res_opc_cr_by_age        = finalize_and_save(extract_pair(res_by_age, "OPC", "Cajal-Retzius"),
                                              "liana_opc_cr_by_age.csv")
res_neuro_gc_by_age      = finalize_and_save(extract_pair(res_by_age, "Neurogenic_lineage", "GC"),
                                              "liana_neuro_gc_by_age.csv")

# sanity check: each should have rows, both interaction directions, all 3 ages
for name, df in [("astro_vasc", res_astro_vasc_by_age),
                  ("vasc_microglia", res_vasc_microglia_by_age),
                  ("opc_cr", res_opc_cr_by_age),
                  ("neuro_gc", res_neuro_gc_by_age)]:
    print(name, "rows:", len(df), "| sources:", df["source"].unique(), "| ages:", df["age"].unique())


astro_vasc rows: 386 | sources: ['Vascular' 'Astrocyte'] | ages: ['young' 'middle' 'old']
vasc_microglia rows: 325 | sources: ['Vascular' 'Microglia'] | ages: ['young' 'middle' 'old']
opc_cr rows: 540 | sources: ['OPC' 'Cajal-Retzius'] | ages: ['young' 'middle' 'old']
neuro_gc rows: 846 | sources: ['Neurogenic_lineage' 'GC'] | ages: ['young' 'middle' 'old']


In [13]:
res_neuro_gc_by_age[res_neuro_gc_by_age["ligand_complex"] == "Adam10"][res_neuro_gc_by_age["receptor_complex"] == "Notch2"]

/tmp/ipykernel_7033/1618079730.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,lr_gmeans,gmean_pvals,specificity_rank,magnitude_rank,age
3087,GC,Neurogenic_lineage,Adam10,Notch2,0.366591,0.026,0.363985,0.000,0.752029,0.896433,young
7692,GC,Neurogenic_lineage,Adam10,Notch2,0.381394,0.379,0.370462,0.115,1.000000,0.854986,middle
8240,Neurogenic_lineage,GC,Adam10,Notch2,0.185403,1.000,0.145433,1.000,1.000000,1.000000,middle
12476,GC,Neurogenic_lineage,Adam10,Notch2,0.287467,0.923,0.283717,0.240,1.000000,1.000000,old
